# P15A — Audit validasi, sensitivitas, dan klaim

Jalankan sesudah P14A/P14B. Data lereng sepenuhnya sintetis. Split kelompok berulang adalah uji sensitivitas yang saling tumpang tindih, bukan replikasi independen. Bootstrap mengulang kelompok pada satu holdout terpilih dan bersifat kondisional. Permutation importance hanya menunjukkan perilaku prediktif model; bukan sebab atau hukum fisik.

In [1]:
from pathlib import Path
import numpy as np,pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score
from sklearn.inspection import permutation_importance
SEED=15015; rng=np.random.default_rng(SEED)
kandidat=[Path('praktikum/bab15/slope_campaign_synthetic.csv'),Path('praktikum/bab14/slope_campaign_synthetic.csv'),Path('slope_campaign_synthetic.csv')]
DATA=next((p for p in kandidat if p.exists()),None)
if DATA is None:
    raise FileNotFoundError('slope_campaign_synthetic.csv tidak ditemukan. Jalankan P14A dari akar repositori, atau salin berkas itu ke praktikum/bab15/.')
print('Data:',DATA)
df=pd.read_csv(DATA)
features=['slope_angle_deg_synth','cohesion_kPa_synth','drainage_score_synth','vegetation_cover_pct_synth','rain_7d_mm_synth','pore_pressure_index_synth','soil_state_index_synth','campaign_index']
target='deformation_response_synth_mm';X=df[features];y=df[target];groups=df.slope_id
splits=GroupShuffleSplit(n_splits=12,test_size=.25,random_state=SEED); rows=[]; chosen=None
for k,(tr,te) in enumerate(splits.split(X,y,groups),1):
 m=RandomForestRegressor(n_estimators=80,min_samples_leaf=3,max_features=.8,random_state=SEED+k,n_jobs=1);m.fit(X.iloc[tr],y.iloc[tr]);pred=m.predict(X.iloc[te]);base=np.repeat(y.iloc[tr].mean(),len(te))
 rows.append([k,groups.iloc[tr].nunique(),groups.iloc[te].nunique(),mean_absolute_error(y.iloc[te],base),mean_absolute_error(y.iloc[te],pred),mean_squared_error(y.iloc[te],pred)**.5,r2_score(y.iloc[te],pred)])
 if k==1:chosen=(tr,te,m,pred,base)
scores=pd.DataFrame(rows,columns=['repeat','groups_train','groups_test','baseline_MAE','model_MAE','model_RMSE','model_R2']);out=Path('praktikum/bab15');scores.to_csv(out/'P15_repeated_group_splits.csv',index=False)
print('Repeated group split summary (overlapping splits; sensitivity, not independent CI)')
print(scores[['baseline_MAE','model_MAE','model_RMSE','model_R2']].describe(percentiles=[.1,.5,.9]).round(4))
tr,te,m,pred,base=chosen
predtab=df.iloc[te][['slope_id','campaign_index',target]].copy();predtab=predtab.rename(columns={target:'actual_synth'});predtab['predicted_synth']=pred;predtab['residual_synth']=predtab.actual_synth-predtab.predicted_synth;predtab['abs_error_synth']=predtab.residual_synth.abs();predtab.to_csv(out/'P15_holdout_predictions.csv',index=False)
# Cluster bootstrap: resample 18 held-out slopes as blocks, conditional on this split/model.
ids=predtab.slope_id.unique();boot=[]
for b in range(500):
 draw=rng.choice(ids,size=len(ids),replace=True);idx=np.concatenate([np.flatnonzero(predtab.slope_id.to_numpy()==sid) for sid in draw]);boot.append([mean_absolute_error(predtab.actual_synth.iloc[idx],predtab.predicted_synth.iloc[idx]),mean_absolute_error(predtab.actual_synth.iloc[idx],np.repeat(y.iloc[tr].mean(),len(idx)))])
boot=np.asarray(boot); ci=pd.DataFrame({'quantity':['model_MAE','baseline_MAE','baseline_minus_model_MAE'],'p2_5':[np.quantile(boot[:,0],.025),np.quantile(boot[:,1],.025),np.quantile(boot[:,1]-boot[:,0],.025)],'median':[np.median(boot[:,0]),np.median(boot[:,1]),np.median(boot[:,1]-boot[:,0])],'p97_5':[np.quantile(boot[:,0],.975),np.quantile(boot[:,1],.975),np.quantile(boot[:,1]-boot[:,0],.975)]});ci.to_csv(out/'P15_group_bootstrap_conditional.csv',index=False);print('Conditional group bootstrap');print(ci.round(4).to_string(index=False))
# Permutation importance on this same holdout; correlated features make rankings unstable/non-causal.
pi=permutation_importance(m,X.iloc[te],y.iloc[te],scoring='neg_mean_absolute_error',n_repeats=8,random_state=SEED,n_jobs=1);imp=pd.DataFrame({'feature':features,'mae_increase_mean':pi.importances_mean,'mae_increase_sd':pi.importances_std}).sort_values('mae_increase_mean',ascending=False);imp.to_csv(out/'P15_permutation_importance_synth.csv',index=False);print('Permutation importance on one held-out group set; predictive association only');print(imp.round(4).to_string(index=False))
# Orange inspection table
cols=['slope_id','campaign_index','actual_synth','predicted_synth','residual_synth','abs_error_synth'];vals=predtab[cols];tab=out/'P15_holdout_orange.tab';types=['string','continuous','continuous','continuous','continuous','continuous'];flags=['meta','meta','class','','','']
with tab.open('w',encoding='utf-8',newline='') as f:f.write('\t'.join(cols)+'\n'+'\t'.join(types)+'\n'+'\t'.join(flags)+'\n');vals.to_csv(f,sep='\t',index=False,header=False)
print('rows',len(df),'groups',groups.nunique(),'chosen test groups',len(ids),'synthetic only')

Data: praktikum/bab15/slope_campaign_synthetic.csv


Repeated group split summary (overlapping splits; sensitivity, not independent CI)
       baseline_MAE  model_MAE  model_RMSE  model_R2
count       12.0000    12.0000     12.0000   12.0000
mean         0.7681     0.7400      0.9563   -0.3215
std          0.0734     0.0853      0.1159    0.2931
min          0.6251     0.5792      0.7593   -0.9081
10%          0.6978     0.6565      0.8231   -0.6038
50%          0.7720     0.7315      0.9479   -0.3091
90%          0.8448     0.8416      1.1033   -0.1316
max          0.8844     0.8630      1.1482    0.3041


Conditional group bootstrap
                quantity    p2_5  median  p97_5
               model_MAE  0.4883  0.7396 1.0184
            baseline_MAE  0.6382  0.7679 0.8940
baseline_minus_model_MAE -0.1942  0.0338 0.2428


Permutation importance on one held-out group set; predictive association only
                   feature  mae_increase_mean  mae_increase_sd
vegetation_cover_pct_synth             0.0452           0.0167
    soil_state_index_synth             0.0179           0.0158
 pore_pressure_index_synth             0.0098           0.0031
          rain_7d_mm_synth             0.0064           0.0020
            campaign_index             0.0003           0.0009
        cohesion_kPa_synth            -0.0121           0.0033
     slope_angle_deg_synth            -0.0185           0.0075
      drainage_score_synth            -0.0571           0.0117
rows 1296 groups 72 chosen test groups 18 synthetic only
